In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import numpy as np

# Paths to the data inside the Kaggle environment
TRAIN_PATH = "/kaggle/input/petfinder-pawpularity-score/train.csv"
TEST_PATH = "/kaggle/input/petfinder-pawpularity-score/test.csv"

# ------------------------------------------------------------------
# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# Features used in the original code (metadata columns)
metadata_features = [
    "Subject Focus",
    "Eyes",
    "Face",
    "Near",
    "Action",
    "Accessory",
    "Group",
    "Collage",
    "Human",
    "Occlusion",
    "Info",
    "Blur",
]

X = train_df[metadata_features].values
y = train_df["Pawpularity"].values

# Simple train/validation split to get an idea of performance
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ------------------------------------------------------------------
# Train XGBoost regressor
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    n_jobs=4,
    random_state=42,
    reg_lambda=1.0,
)

model.fit(X_train, y_train)

# Optional: evaluate on validation set (RMSE)
val_pred = model.predict(X_val)
val_rmse = mean_squared_error(y_val, val_pred, squared=False)
print(f"Validation RMSE: {val_rmse:.4f}")

# ------------------------------------------------------------------
# Predict on the test set
X_test = test_df[metadata_features].values
test_preds = model.predict(X_test)

# Pawpularity scores are bounded between 0 and 100; enforce this
test_preds = np.clip(test_preds, 0, 100)

# ------------------------------------------------------------------
# Create submission DataFrame with correct column names
submission = pd.DataFrame({"Id": test_df["Id"], "Pawpularity": test_preds})

# Write to CSV – Kaggle expects the file in the working directory
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")



Validation RMSE: 21.6285
Submission written to submission.csv


In [2]:
# This cell is intentionally left empty; all work is performed in cell 1.